# B2.9 · Remediation engineering

**Function B — Application Security with an AI SDLC → The AI SDLC: Harnesses and an Agentic AppSec Pipeline**  ·  *AI for Security*

Builds on **[B2.8 · Exploit chaining](https://spbreed.github.io/cyber-commons/lessons/B2.8.html)**.

| | |
|---|---|
| Tools used | Semgrep OSS, pytest, GLM-4.6, Kimi K2, Claude Sonnet 5 |

## What this lesson is

**What it covers.** Validate four candidate patches on three axes and show which of them only made the scanner green.

**Why a security engineer needs it.** A patch that silences the scanner is indistinguishable from a patch that fixes the bug. The control it builds is: stage 14: generate the fix, re-run the exploit against the patched build, and require a regression test.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

A patch that passes the tests and changes the behaviour is not a fix, it is a second incident with a pull request attached. Remediation is the stage where the pipeline stops finding things and starts touching them.

> **At CyberTravels.** The Coding Agent's fix must not break booking behaviour. A patch that passes the tests and changes what travellers experience is a second incident with a pull request attached. R8.

## 2 · The framework

```
   patch                    what has to be true
   +----------------+       +-----------------------------+
   | fixes the bug  |  and  | behaviour unchanged         |
   |                |       | tests still pass            |
   |                |       | reviewer can follow the why |
   +----------------+       +-----------------------------+

   a patch that passes the tests and changes the behaviour is
   a second incident with a pull request attached
```

**Stage 14 — Remediation engineering.** Generate the fix, then prove it.

A model that finds bugs is useful. A model that fixes them is only useful if you
can tell a real fix from a plausible one, and plausible is exactly what language
models are optimised to produce.

There are three ways to make a finding stop firing:

1. **Fix the vulnerability** — behaviour preserved, bug gone.
2. **Remove the code** — finding gone, so is the feature.
3. **Evade the detector** — rewrite until the pattern misses.

All three make the scanner green, and an autonomous loop optimising for a green
scan will find options 2 and 3 on its own because they are cheaper.

The pipeline has an advantage a static workflow does not: Phase 4 already built
a working exploit. So the acceptance test is not "does the scanner still fire?"
It is **"does the exploit still work against the patched build?"** — which is
the only question that cannot be gamed by editing the code around the detector.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · The stage, as a skill

Several candidate patches make the scanner green; one of them is a fix. The skill runs all three gates — behaviour unchanged, exploit blocked, and proof of fix against the old build — and reports which gate each rejected candidate died at.

In [ ]:
# skills/appsec/patch-validation-harness/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: patch-validation-harness
description: >-
  Accept a proposed fix only when three things hold — behaviour unchanged, the
  exploit no longer works, and the fix proved against the build that was
  vulnerable. Use when an agent proposes a patch, or when a scanner going green
  is being read as a fix.
allowed-tools: Read, Grep, Glob, Bash
---

# A green scanner is not a fixed bug

Several candidate patches will make the scanner green. Some of them change
behaviour, some of them leave the bug exploitable, and one of them does neither.
Telling them apart needs three gates, and the third is the one that is usually
missing: proof against the **old** build, so "the exploit stops working" is a
statement about the patch rather than about the environment.

## When to use this

Every proposed remediation, whether authored by a person or an agent, and
especially when the evidence offered is that the scanner no longer fires.

## Procedure

**1 — Establish the baseline on the vulnerable build.** The behaviour cases must
pass and the exploit must work. If the exploit does not work here, you are about
to validate a patch against a bug you have not reproduced.

**2 — Gate one: behaviour unchanged.** Run every behaviour case against the
patched build. A patch that fixes the defect and changes an answer is a
regression with a security justification.

**3 — Gate two: the exploit no longer works.** Against the patched build,
directly. Not "the scanner is quiet" — the scanner was one of the tools that
missed the defect class in the first place.

**4 — Gate three: proof of fix.** Run the exploit against the old build again,
after the patch is written, in the same harness. It must still work. This is
what excludes the environment having changed underneath the test.

**5 — Report per candidate and per gate.** A candidate rejected at gate one and
one rejected at gate two need different conversations with whoever wrote them.

## Output contract

```json
{
  "baseline": {"behaviour_pass": true, "exploit_works": true},
  "candidates": [{"id": "str", "scanner_green": true,
                  "behaviour_unchanged": true, "exploit_blocked": true,
                  "proof_of_fix": true, "verdict": "accepted|rejected", "rejected_at": "str|null"}],
  "accepted": ["str"]
}
```

## Failure modes

- **Accepting a green scanner.** Several wrong patches produce one.
- **Skipping the behaviour cases.** The most reliable way to block an exploit is
  to break the feature.
- **Omitting proof of fix.** Without it you cannot distinguish a working patch
  from a broken exploit.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/appsec/patch-validation-harness/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/appsec/patch-validation-harness/scripts/patch_validation_harness.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Accept a patch only when behaviour is unchanged, the exploit stops working, and the fix is proved against the old build.

This is the executable half of the `patch-validation-harness` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

# --- model backend: replay by default, a Kaggle open-weight model when served -
# One URL and one header shape, no vendor SDK. Standard library only, so the
# notebook stays self-contained.
import json, os, urllib.error, urllib.request

# Qwen2.5-7B-Instruct is the floor established in MODELS.md: below it two of
# the lessons' acceptance properties stop holding.
OPEN_WEIGHT_DEFAULT = "qwen2.5-7b-instruct"
TIMEOUT = 60

def backend():
    """(kind, model). Configuration comes from the environment, never a literal."""
    if os.environ.get("OPENAI_BASE_URL"):
        return "open-weight", os.environ.get("MODEL", OPEN_WEIGHT_DEFAULT)
    return "replay", "deterministic stand-in (no backend configured)"

def _post(url, payload, headers):
    req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                 headers={"content-type": "application/json", **headers})
    with urllib.request.urlopen(req, timeout=TIMEOUT) as r:
        return json.loads(r.read().decode())

def _openai_compatible(prompt, system, model, max_tokens, temperature):
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": prompt}]
    base = os.environ["OPENAI_BASE_URL"].rstrip("/")
    key = os.environ.get("OPENAI_API_KEY", "not-needed")
    out = _post(f"{base}/chat/completions",
                {"model": model, "messages": msgs, "max_tokens": max_tokens,
                 "temperature": temperature},
                {"authorization": f"Bearer {key}"})
    return out["choices"][0]["message"]["content"].strip()

def ask(prompt, *, replay, system=None, max_tokens=512, temperature=0.0):
    """Answer `prompt` with the configured backend, or return `replay`.

    `replay` is required, not optional: a lesson must be able to run offline,
    and the answer it falls back to has to be visible in the source rather than
    invented at runtime.
    """
    kind, model = backend()
    if kind == "replay":
        return replay, kind, model
    try:
        return _openai_compatible(prompt, system, model, max_tokens,
                                  temperature), kind, model
    except (urllib.error.URLError, urllib.error.HTTPError, KeyError, TimeoutError) as e:
        # Print what the server actually said. "failed: 400" costs whoever hits
        # this an hour; the body usually names the exact missing parameter, and
        # it never contains a key.
        detail = getattr(e, "code", None) or type(e).__name__
        why = ""
        if hasattr(e, "read"):
            try:
                why = json.loads(e.read().decode()).get("error", {}).get("message", "")
            except Exception:
                why = ""
        print(f"   !! {kind} backend ({model}) failed: {detail}"
              f"{' - ' + why if why else ''}")
        print("      Using the replay, which is labelled as one. No model answered.")
        return replay, "replay", f"{model} unreachable"

_kind, _model = backend()
print(f"model backend : {_kind}")
print(f"model         : {_model}")
if _kind == "replay":
    print()
    print("This lesson runs offline against a deterministic replay, which is why")
    print("it works on a Kaggle kernel with the internet switched off. To run the")
    print("identical code against a real model, serve an open-weight model from")
    print("Kaggle Models and point the adapter at it:")
    print()
    print("   python3 -m llama_cpp.server --model <the .gguf from Kaggle> \\")
    print("           --model_alias qwen2.5-7b-instruct --port 11434 --chat_format qwen")
    print("   export OPENAI_BASE_URL=http://127.0.0.1:11434/v1 \\")
    print("          MODEL=qwen2.5-7b-instruct")
    print()
    print("   MODELS.md has the exact Kaggle download. There is no paid backend:")
    print("   every model result in this repository was produced this way.")


import re, sqlite3

VULNERABLE = '''
def get_user(conn, name):
    return conn.execute("SELECT id, name FROM users WHERE name = '" + name + "'").fetchall()
'''

def build_db():
    conn = sqlite3.connect(":memory:")
    conn.execute("CREATE TABLE users(id INTEGER, name TEXT)")
    conn.executemany("INSERT INTO users VALUES (?,?)",
                     [(1,"dana"),(2,"sam"),(3,"o'brien")])
    return conn

def load(src):
    ns = {}; exec(compile(src, "<patch>", "exec"), ns); return ns["get_user"]

BEHAVIOUR = [("dana",[(1,"dana")]), ("sam",[(2,"sam")]),
             ("nobody",[]), ("o'brien",[(3,"o'brien")])]

def behaviour_ok(fn):
    conn = build_db(); rows = []
    for name, expected in BEHAVIOUR:
        try: got = fn(conn, name)
        except Exception as e: rows.append((name, f"raised {type(e).__name__}", False)); continue
        rows.append((name, got, got == expected))
    return rows

def exploit_works(fn):
    """The stage-12 probe, reused as the acceptance test."""
    conn = build_db()
    try: rows = fn(conn, "x' OR '1'='1")
    except Exception: return False, "probe raised — not exploitable this way"
    return len(rows) > 1, f"probe returned {len(rows)} rows"

def scanner_fires(src):
    return bool(re.search(r"execute\(\s*[\"\'][^\"\']*[\"\']\s*\+", src))

fn = load(VULNERABLE)
print("behaviour of the vulnerable build:")
for name, got, ok in behaviour_ok(fn):
    print(f"   get_user({name!r:10s}) → {str(got):18s} {'ok' if ok else 'FAILS'}")
ex, why = exploit_works(fn)
print(f"\nexploit works: {ex} — {why}")
print(f"scanner fires: {scanner_fires(VULNERABLE)}")

CANDIDATES = {
 "A · parameterise (the real fix)": '''
def get_user(conn, name):
    return conn.execute("SELECT id, name FROM users WHERE name = ?", (name,)).fetchall()
''',
 "B · delete the feature": '''
def get_user(conn, name):
    return []
''',
 "C · evade the scanner": '''
def get_user(conn, name):
    q = "SELECT id, name FROM users WHERE name = '%s'" % name
    return conn.execute(q).fetchall()
''',
 "D · escape by hand": '''
def get_user(conn, name):
    safe = name.replace("'", "''")
    return conn.execute("SELECT id, name FROM users WHERE name = '" + safe + "'").fetchall()
''',
}
print(f"{'candidate':34s}{'scanner green':>15}")
print("-" * 50)
for name, src in CANDIDATES.items():
    print(f"{name:34s}{str(not scanner_fires(src)):>15}")
print("\nThree of four are green. Only one of those is a fix.")

def validate(src):
    fn = load(src)
    green = not scanner_fires(src)
    beh = behaviour_ok(fn)
    preserved = all(ok for _, _, ok in beh)
    still_exploitable, _ = exploit_works(fn)
    reasons = []
    if not green:            reasons.append("scanner still fires")
    if not preserved:        reasons.append("behaviour changed")
    if still_exploitable:    reasons.append("STILL EXPLOITABLE (stage-12 probe passes)")
    return (not reasons), green, preserved, still_exploitable, reasons

print(f"{'candidate':34s}{'scan':6s}{'behaviour':11s}{'exploitable':13s}verdict")
print("-" * 84)
accepted = []
for name, src in CANDIDATES.items():
    ok, g, b, x, reasons = validate(src)
    if ok: accepted.append(name)
    print(f"{name:34s}{str(g):6s}{str(b):11s}{str(x):13s}"
          f"{'ACCEPT' if ok else 'REJECT — ' + ', '.join(reasons)}")
print(f"\naccepted: {accepted}")
assert "A · parameterise (the real fix)" in accepted
assert "B · delete the feature" not in accepted
assert "C · evade the scanner" not in accepted

# The proof-of-fix clause: the exploit must fail on the new build and
# succeed on the old one. Without both halves, "fixed" is a claim.
def proof_of_fix(old_src, new_src):
    old_ex, _ = exploit_works(load(old_src))
    new_ex, _ = exploit_works(load(new_src))
    return (old_ex and not new_ex), f"exploit on old={old_ex}, on new={new_ex}"

for name in accepted:
    ok, detail = proof_of_fix(VULNERABLE, CANDIDATES[name])
    print(f"{name:34s} proof of fix: {ok}  ({detail})")

print("\nCandidate D passes every automated check and is still the wrong answer:")
print("it reimplements the driver's escaping and will be wrong for the next")
print("input class or the next database. Nothing except a rule about MECHANISM")
print("catches that — which is the part of remediation that does not automate.")

# ------------------------------------ the same task, against a real model
# Offline this is a labelled replay; with an open-weight model served
# from Kaggle it is the same code calling a real one.

TASK = 'Fix this without changing the function\'s behaviour for valid input. Return only the patched function.\n\ndef report(request):\n    q = "SELECT * FROM orders WHERE ref = \'" + request.args[\'ref\'] + "\'"\n    return db.execute(q)'

REPLAY = 'def report(request):\n    q = "SELECT * FROM orders WHERE ref = ?"\n    return db.execute(q, (request.args[\'ref\'],))'

answer, used, model = ask(TASK, replay=REPLAY,
            system='You are a remediation engineer. Output code only, no explanation.',
            max_tokens=300)

print(f"backend used : {used}")
print(f"model        : {model}")
print(f"prompt       : {TASK[:66]}...")
print()
print("answer:")
for line in (answer.splitlines() or [answer]):
    print(f"   {line}")

# Two assertions that must hold on every backend, and one property that is
# reported rather than asserted - a real model failing it is a finding about
# the model, not a broken notebook.
assert answer.strip(), "the configured backend returned nothing"
if used == "replay":
    assert answer == REPLAY, "the offline path must return the replay verbatim"

label, held = ("parameterises the query", "?" in answer or "%s" in answer or ":ref" in answer)
print()
print(f"property checked : {label}")
print(f"held on {used:12s} : {held}")
print()
print("Same code, same assertions, two possible backends. Offline the answer is")
print("the replay and is labelled as one; with a served model it is the model's.")

## What you just proved

The vulnerable build passes all four behaviour cases and the exploit returns 3 rows. Candidates A, B and D make the scanner green. Validation rejects B for changed behaviour and C for remaining exploitable, accepting A and D. Proof of fix holds for both accepted patches — the exploit works on the old build and fails on the new.

## Your turn

Candidate D passes every automated gate and is still wrong. Write the rule that rejects it. You will find it has to be about which *mechanism* is acceptable, not about outcomes — and that rule belongs in your secure coding standard, not in the pipeline.

---

**Next → [B2.10 · Severity calibration and reporting](https://spbreed.github.io/cyber-commons/lessons/B2.10.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.9.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.9.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*